# Notebook 1 — Data Cleaning and Translation

**Project:** Kosovo Labor Market Demand Analysis 2025  
**Data source:** SuperPuna.com  
**Description:** This notebook loads the raw SuperPuna job postings dataset, normalizes Albanian job titles by removing gendered suffixes, translates unique titles from Albanian to English using the Google Translate free tier, and saves the cleaned dataset to Google Drive for use in Notebook 2.

**Output:** `superpuna_cleaned.csv` saved to Google Drive  
**Next step:** Run `02_esco_standardization.ipynb`

---

## 1. Install and Import Libraries

In [ ]:
# Install external libraries not available by default in Colab
!pip install deep-translator tqdm openpyxl -q

In [ ]:
# Standard library
import re
import os
import time

# Data
import pandas as pd

# Translation
from deep_translator import GoogleTranslator
from tqdm.notebook import tqdm

# Colab utilities
from google.colab import files, drive

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 2. Constants

In [ ]:
# Column names — defined once here so any change propagates through the notebook
TITLE_COL    = 'Titulli_i_vendit_te_punes'
VACANCY_COL  = 'Numri_i_vendeve_te_lira'
COMPANY_COL  = 'Kompania'
CITY_COL     = 'Vendi_i_punes'
SALARY_COL   = 'Paga'
DATE_COL     = 'Data_e_shpalljes'

# Translation settings
# Sleep between requests avoids hitting Google Translate rate limits
TRANSLATION_SLEEP   = 0.5
CHECKPOINT_INTERVAL = 500
SOURCE_LANGUAGE     = 'sq'   # ISO 639-1 code for Albanian
TARGET_LANGUAGE     = 'en'

# Google Drive path — all intermediate files saved here
DRIVE_DIR = '/content/drive/MyDrive/kosovo_lm'

print('Constants defined.')

## 3. Mount Google Drive

In [ ]:
# Mount Google Drive so files persist between sessions
drive.mount('/content/drive')

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Google Drive mounted. Working directory: {DRIVE_DIR}')

## 4. Load Dataset

In [ ]:
# Upload the raw SuperPuna dataset
# Accepts both CSV and Excel formats
print('Upload your SuperPuna dataset (CSV or Excel)...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
elif filename.endswith('.csv'):
    loaded = False
    # Try common encodings and separators — scraped CSVs are often inconsistent
    for encoding in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(filename, sep=sep, encoding=encoding,
                                 on_bad_lines='skip', engine='python')
                if df.shape[1] >= 3:
                    print(f'Loaded with encoding={encoding}, separator="{sep}"')
                    loaded = True
                    break
            except Exception:
                continue
        if loaded:
            break

# Parse date and salary columns
df[DATE_COL]   = pd.to_datetime(df[DATE_COL], errors='coerce')
df[SALARY_COL] = pd.to_numeric(df[SALARY_COL], errors='coerce')
df[VACANCY_COL]= pd.to_numeric(df[VACANCY_COL], errors='coerce').fillna(1)

print(f'Dataset loaded: {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 5. Normalize Albanian Job Titles

In [ ]:
def normalize_albanian_title(title):
    """
    Normalize an Albanian job title to remove gendered variants
    and diacritic characters that create artificial duplicates.

    Albanian uses both slash notation (Shites/e) and backslash
    notation (Shites\\e) to indicate male/female variants of the
    same role. These are collapsed to a single form before translation
    to reduce the number of unique titles that need to be processed.

    Examples:
        'Kamarier/e'  -> 'Kamarier'
        'Shites\\e'   -> 'Shites'
        'Arkatar / E' -> 'Arkatar'
        'Infermier(e)'-> 'Infermier'
    """
    if not isinstance(title, str):
        return ''
    t = title.strip()
    # Normalize backslash to forward slash before applying regex
    t = t.replace('\\', '/')
    # Remove gender slash patterns: /e, /ja, /i, / E etc.
    t = re.sub(r'\s*/\s*[eEjaJAiI]{1,2}\b', '', t)
    # Remove trailing slash with nothing after
    t = re.sub(r'\s*/\s*$', '', t)
    # Remove parenthetical gender markers: (e), (i), (ja)
    t = re.sub(r'\s*\([eEiIjJ]{1,2}\)', '', t)
    # Standardize Albanian diacritics
    t = t.replace('e', 'e').replace('E', 'E')
    t = t.replace('c', 'c').replace('C', 'C')
    t = re.sub(r'\s+', ' ', t).strip()
    return t


df['title_normalized'] = df[TITLE_COL].apply(normalize_albanian_title)

before = df[TITLE_COL].nunique()
after  = df['title_normalized'].nunique()

print('Title normalization complete.')
print(f'Unique titles before normalization : {before:,}')
print(f'Unique titles after normalization  : {after:,}')
print(f'Variants removed                   : {before - after:,}')

## 6. Detect Language — Separate Albanian and English Titles

In [ ]:
def is_likely_english(text):
    """
    Heuristic check to identify titles already written in English.
    Titles identified as English are excluded from translation to
    reduce API calls and avoid introducing translation errors.
    """
    if not isinstance(text, str):
        return False
    english_keywords = [
        'manager', 'engineer', 'developer', 'analyst', 'designer',
        'coordinator', 'specialist', 'consultant', 'director', 'officer',
        'assistant', 'supervisor', 'technician', 'accountant', 'sales',
        'marketing', 'finance', 'hr ', 'it ', 'ceo', 'cfo', 'cto',
        'operator', 'intern', 'senior', 'junior', 'lead', 'head of',
        'driver', 'nurse', 'teacher', 'chef', 'cook', 'security'
    ]
    return any(kw in text.lower() for kw in english_keywords)


unique_titles     = [t for t in df['title_normalized'].dropna().unique() if t.strip()]
already_english   = [t for t in unique_titles if is_likely_english(t)]
needs_translation = [t for t in unique_titles if not is_likely_english(t)]

print(f'Total unique normalized titles : {len(unique_titles):,}')
print(f'Already English (skip)         : {len(already_english):,}')
print(f'Requires translation           : {len(needs_translation):,}')

## 7. Translate Albanian Titles to English

In [ ]:
def translate_with_retry(translator, title, retries=3):
    """
    Translate a single title with exponential backoff on failure.
    Falls back to the original title if all retries are exhausted.
    """
    for attempt in range(retries):
        try:
            result = translator.translate(title)
            return result if result else title
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return title


def translate_batch(titles, src=SOURCE_LANGUAGE, target=TARGET_LANGUAGE,
                    batch_size=50):
    """
    Translate a list of titles using the Google Translate free tier.
    Saves a checkpoint to Google Drive every CHECKPOINT_INTERVAL titles
    so progress is not lost if the session disconnects.
    """
    translation_map = {}
    translator = GoogleTranslator(source=src, target=target)

    # Resume from existing checkpoint if available
    checkpoint_path = f'{DRIVE_DIR}/translation_map.csv'
    if os.path.exists(checkpoint_path):
        saved = pd.read_csv(checkpoint_path)
        translation_map = dict(zip(saved['albanian'], saved['english']))
        titles = [t for t in titles if t not in translation_map]
        print(f'Resumed from checkpoint: {len(translation_map):,} already translated, '
              f'{len(titles):,} remaining.')

    for i in tqdm(range(0, len(titles), batch_size), desc='Translating'):
        batch = titles[i:i+batch_size]
        for title in batch:
            translation_map[title] = translate_with_retry(translator, title)

        if len(translation_map) % CHECKPOINT_INTERVAL == 0:
            pd.DataFrame(translation_map.items(),
                         columns=['albanian', 'english']
                        ).to_csv(checkpoint_path, index=False)
            print(f'Checkpoint saved: {len(translation_map):,} titles translated.')

        time.sleep(TRANSLATION_SLEEP)

    return translation_map


print(f'Translating {len(needs_translation):,} unique titles from Albanian to English...')
print('Estimated time: approximately 45 minutes. Checkpoints saved every '
      f'{CHECKPOINT_INTERVAL} titles.\n')

translation_map = translate_batch(needs_translation)

for t in already_english:
    translation_map[t] = t

# Final save
pd.DataFrame(translation_map.items(),
             columns=['albanian', 'english']
            ).to_csv(f'{DRIVE_DIR}/translation_map.csv', index=False)

print(f'\nTranslation complete: {len(translation_map):,} titles.')
print(f'Translation map saved to: {DRIVE_DIR}/translation_map.csv')

## 8. Manual Translation Corrections

In [ ]:
# Some short Albanian words are mistranslated out of context.
# These corrections are applied after automatic translation.
# Add additional entries as needed after reviewing translation output.
manual_fixes = {
    'Workplace'        : 'Worker',
    'paymaster'        : 'Cashier',
    'steward'          : 'Waiter',
    'Steward'          : 'Waiter',
    'Agent'            : 'Field Agent',
}

fixed_count = 0
for albanian, english in translation_map.items():
    if english in manual_fixes:
        translation_map[albanian] = manual_fixes[english]
        fixed_count += 1

print(f'Manual corrections applied: {fixed_count}')

# Save corrected map
pd.DataFrame(translation_map.items(),
             columns=['albanian', 'english']
            ).to_csv(f'{DRIVE_DIR}/translation_map.csv', index=False)

## 9. Add English Titles to Dataset

In [ ]:
df['title_english'] = df['title_normalized'].map(translation_map)
# Where no translation exists, retain the normalized Albanian title
df['title_english'] = df['title_english'].fillna(df['title_normalized'])

print('English titles added to dataset.')
print(f'Rows with English title: {df["title_english"].notna().sum():,}')
df[[TITLE_COL, 'title_normalized', 'title_english']].head(10)

## 10. Save Cleaned Dataset to Google Drive

In [ ]:
output_path = f'{DRIVE_DIR}/superpuna_cleaned.csv'
df.to_csv(output_path, index=False)

print('Cleaned dataset saved successfully.')
print(f'Path     : {output_path}')
print(f'Rows     : {len(df):,}')
print(f'Columns  : {len(df.columns)}')
print('\nNext step: open 02_esco_standardization.ipynb')